# Lab 3 : A simple AWS agent

*Week 5 · Utrains LLMOps 8 Week Course*

Run each cell from the top. Read what it prints before you run the next cell.

**(Reload this file from disk if you still see the old "when agents fail" lab.)**


## Objective

Labs 1 and 2 used a small **order-desk** example (mock data in Python).

This lab uses the **same agent idea** on a real system: **Amazon Web Services (AWS)**.

You will build three tools:

| Tool | What it answers |
|------|-----------------|
| `get_ec2_instance` | Is this EC2 instance present? What is its state? |
| `get_s3_bucket` | Does this S3 bucket exist? |
| `get_iam_user` | Does this IAM user exist? |

Then a user asks a question in plain English. The model **picks the right tool**, your code **runs it against AWS**, and the model **answers from the tool result**.

That is an agentic system: **question → select tool → run tool → answer**.

### What you need

1. Same `week05/.env` with `ANTHROPIC_API_KEY`.
2. Also add your AWS keys (see Step 1).
3. Use **your** instance id, bucket name, and IAM user name when you ask questions.

**Safety.** Prefer an IAM user with **read-only** access (describe EC2, check S3, get IAM user). Never commit `.env`.


### Step 1. Load keys, model, and AWS clients

`python-dotenv` loads secrets from `.env`.

`boto3` is the official AWS SDK for Python. It reads:

- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`

Put those in `week05/.env` next to your Anthropic key.


In [ ]:
import os

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
import boto3

load_dotenv()

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# boto3 uses the AWS keys from .env
ec2 = boto3.client("ec2")
s3 = boto3.client("s3")
iam = boto3.client("iam")

print("Model ready.")
print("AWS region:", os.getenv("AWS_DEFAULT_REGION", "(not set — add AWS_DEFAULT_REGION to .env)"))
print("AWS access key loaded:", bool(os.getenv("AWS_ACCESS_KEY_ID")))


### Step 2. Three simple AWS tools

Each tool is a normal Python function. The docstring tells the model **when** to use it.

We keep the bodies short on purpose. The lesson is tool selection + the agent path — not advanced AWS code.


In [ ]:
from langchain_core.tools import tool


@tool
def get_ec2_instance(instance_id: str) -> dict:
    """Check if an EC2 instance exists. Returns state and type when found."""
    try:
        response = ec2.describe_instances(InstanceIds=[instance_id])
        reservations = response.get("Reservations", [])
        if not reservations or not reservations[0].get("Instances"):
            return {"exists": False, "instance_id": instance_id}
        instance = reservations[0]["Instances"][0]
        return {
            "exists": True,
            "instance_id": instance_id,
            "state": instance["State"]["Name"],
            "instance_type": instance.get("InstanceType"),
        }
    except Exception as error:
        return {"exists": False, "instance_id": instance_id, "error": str(error)}


@tool
def get_s3_bucket(bucket_name: str) -> dict:
    """Check if an S3 bucket exists."""
    try:
        s3.head_bucket(Bucket=bucket_name)
        return {"exists": True, "bucket": bucket_name}
    except Exception as error:
        return {"exists": False, "bucket": bucket_name, "error": str(error)}


@tool
def get_iam_user(user_name: str) -> dict:
    """Check if an IAM user exists. Returns the user ARN when found."""
    try:
        response = iam.get_user(UserName=user_name)
        user = response["User"]
        return {
            "exists": True,
            "user_name": user["UserName"],
            "arn": user["Arn"],
        }
    except Exception as error:
        return {"exists": False, "user_name": user_name, "error": str(error)}


TOOLS = [get_ec2_instance, get_s3_bucket, get_iam_user]
print("Tools ready:", [t.name for t in TOOLS])


### Step 3. The agent path (same idea as Lab 2)

One cell. Four steps:

1. **SELECT TOOL** — model chooses EC2, S3, or IAM  
2. **RUN TOOL** — your code calls AWS  
3. **SEND RESULT BACK** — tool output goes into the conversation  
4. **FINAL ANSWER** — model answers the user  

Change only the `question` string to try different cases.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm_with_tools = llm.bind_tools(TOOLS)
# --- Change this question to try EC2 / S3 / IAM ---
question = "Is EC2 instance i-0123456789abcdef0 present? What is its state?"

messages = [
    SystemMessage(
        content=(
            "You are an AWS assistant. "
            "Use tools for EC2, S3, and IAM questions. "
            "Answer briefly using only tool results. "
            "Do not invent AWS data."
        )
    ),
    HumanMessage(content=question),
]

# 1) SELECT TOOL
ai = llm_with_tools.invoke(messages)
messages.append(ai)

print("1) SELECT TOOL")
print("   tool_calls:", ai.tool_calls)
print()

if not ai.tool_calls:
    print("Model answered without a tool:")
    print(ai.content)
else:
    # 2) RUN TOOL
    call = ai.tool_calls[0]
    name = call["name"]
    args = call["args"]

    if name == "get_ec2_instance":
        result = get_ec2_instance.invoke(args)
    elif name == "get_s3_bucket":
        result = get_s3_bucket.invoke(args)
    elif name == "get_iam_user":
        result = get_iam_user.invoke(args)
    else:
        result = {"error": f"Unknown tool: {name}"}

    print("2) RUN TOOL")
    print("   name:", call["name"])
    print("   args:", call["args"])
    print("   result:", result)
    print()

    # 3) SEND RESULT BACK
    messages.append(
        ToolMessage(content=str(result), tool_call_id=call["id"])
    )
    print("3) SEND RESULT BACK")
    print()

    # 4) FINAL ANSWER
    final = llm_with_tools.invoke(messages)
    print("4) FINAL ANSWER")
    print("  ", final.content)


### Step 4. Try the other tools

Re-run Step 3 after changing `question` to something like:

- **S3:** `Does the S3 bucket my-bucket-name exist?`
- **IAM:** `Does IAM user alice exist in this account?`

Use real names from **your** AWS account. If the resource is missing, a good agent says it was not found — that is still a correct tool result.

### What you should remember

1. Lab 1: the model **asks** for a tool.  
2. Lab 2: your code **runs** the tool and the model **answers**.  
3. Lab 3: the same pattern, but tools talk to a **real system** (AWS).

The concept does not change. Only the data source changes.
